In [1]:
import requests, pandas as pd
from bs4 import BeautifulSoup

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

# === CHANGE THESE ===
company = "HNST/honest company"           # e.g. "MSFT/microsoft"
metric  = "all"                      # "revenue" | "net-income" | "total-assets" | "total-liabilities" | "share-holder-equity" | "all"
# ====================

def scrape_metric(company, slug, col_key, value_col, file_stub):
    URL = f"https://www.macrotrends.net/stocks/charts/{company}/{slug}"
    headers = {"User-Agent":"Mozilla/5.0"}

    html = requests.get(URL, headers=headers, timeout=30).text
    soup = BeautifulSoup(html, "lxml")

    tables = soup.select("table.historical_data_table.table")
    if not tables:
        raise RuntimeError("No tables with class 'historical_data_table table' found.")

    base = company.split("/")[-1].lower()  # e.g., "microsoft"
    names = [f"{base}_annual_{file_stub}.csv", f"{base}_quarterly_{file_stub}.csv"]

    for tbl, out in zip(tables, names):
        df = pd.read_html(str(tbl))[0]
        df.columns = ["k", col_key]

        # type first column
        if df["k"].astype(str).str.fullmatch(r"\d{4}").all():
            df = df.rename(columns={"k": "year"}).assign(
                **{value_col: df[col_key].str.replace("$","",regex=False).str.replace(",","",regex=False).astype(float)}
            ).drop(columns=[col_key]).sort_values("year")
        else:
            df = df.rename(columns={"k": "date"}).assign(
                **{value_col: df[col_key].str.replace("$","",regex=False).str.replace(",","",regex=False).astype(float)}
            ).drop(columns=[col_key])
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.sort_values("date")

        df.to_csv(out, index=False)
        print("Saved", out)

def run(company, metric):
    if metric == "all":
        scrape_metric(company, "revenue", "revenue", "revenue_musd", "revenue")
        scrape_metric(company, "net-income", "net income", "net_income_musd", "net_income")
        scrape_metric(company, "total-assets", "total assets", "total_assets_musd", "total_assets")
        scrape_metric(company, "total-liabilities", "total liabilities", "total_liabilities_musd", "total_liabilities")
        scrape_metric(company, "total-share-holder-equity", "share holder equity", "share_holder_equity_musd", "share_holder_equity")
        merge_all(company)  # build wide annual & quarterly files
    elif metric == "revenue":
        scrape_metric(company, "revenue", "revenue", "revenue_musd", "revenue")
    elif metric == "net-income":
        scrape_metric(company, "net-income", "net income", "net_income_musd", "net_income")
    elif metric == "total-assets":
        scrape_metric(company, "total-assets", "total assets", "total_assets_musd", "total_assets")
    elif metric == "total-liabilities":
        scrape_metric(company, "total-liabilities", "total liabilities", "total_liabilities_musd", "total_liabilities")
    elif metric == "share-holder-equity":
        scrape_metric(company, "total-share-holder-equity", "share holder equity", "share_holder_equity_musd", "share_holder_equity")
    else:
        raise ValueError("metric must be one of: 'revenue', 'net-income', 'total-assets', 'total-liabilities', 'share-holder-equity', or 'all'.")

def merge_all(company):
    base = company.split("/")[-1].lower()
    # (path, value_column)
    annual_files = [
        (f"{base}_annual_revenue.csv",           "revenue_musd"),
        (f"{base}_annual_net_income.csv",        "net_income_musd"),
        (f"{base}_annual_total_assets.csv",      "total_assets_musd"),
        (f"{base}_annual_total_liabilities.csv", "total_liabilities_musd"),
        (f"{base}_annual_share_holder_equity.csv","share_holder_equity_musd"),
    ]
    quarterly_files = [
        (f"{base}_quarterly_revenue.csv",           "revenue_musd"),
        (f"{base}_quarterly_net_income.csv",        "net_income_musd"),
        (f"{base}_quarterly_total_assets.csv",      "total_assets_musd"),
        (f"{base}_quarterly_total_liabilities.csv", "total_liabilities_musd"),
        (f"{base}_quarterly_share_holder_equity.csv","share_holder_equity_musd"),
    ]

    # Annual merge (wide on 'year')
    merged_a = None
    for path, valcol in annual_files:
        if Path(path).exists():
            t = pd.read_csv(path)
            if "year" in t.columns:
                t = t[["year", valcol]]
                merged_a = t if merged_a is None else pd.merge(merged_a, t, on="year", how="outer")
    if merged_a is not None:
        merged_a = merged_a.sort_values("year")
        out_a = f"{base}_annual_all_metrics.csv"
        merged_a.to_csv(out_a, index=False)
        print("Saved", out_a)
    else:
        print("No annual files found to merge.")

    # Quarterly merge (wide on 'date')
    merged_q = None
    for path, valcol in quarterly_files:
        if Path(path).exists():
            t = pd.read_csv(path, parse_dates=["date"])
            if "date" in t.columns:
                t = t[["date", valcol]]
                merged_q = t if merged_q is None else pd.merge(merged_q, t, on="date", how="outer")
    if merged_q is not None:
        merged_q = merged_q.sort_values("date")
        out_q = f"{base}_quarterly_all_metrics.csv"
        merged_q.to_csv(out_q, index=False)
        print("Saved", out_q)
    else:
        print("No quarterly files found to merge.")

run(company, metric)

/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]


Saved honest company_annual_revenue.csv
Saved honest company_quarterly_revenue.csv


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]


Saved honest company_annual_net_income.csv
Saved honest company_quarterly_net_income.csv


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]


Saved honest company_annual_total_assets.csv
Saved honest company_quarterly_total_assets.csv


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]


Saved honest company_annual_total_liabilities.csv
Saved honest company_quarterly_total_liabilities.csv
Saved honest company_annual_share_holder_equity.csv
Saved honest company_quarterly_share_holder_equity.csv
Saved honest company_annual_all_metrics.csv
Saved honest company_quarterly_all_metrics.csv


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]
/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9346/2732654402.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tbl))[0]
